# VisClick — Colab setup

Run cells **top to bottom**. Authorise Google Drive when prompted.

- **GitHub** → code at `/content/visclick`
- **Drive** `MyDrive/visclick` → datasets + weights (persists after disconnect)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
if not os.path.isdir("/content/visclick"):
    subprocess.run(["git", "clone", REPO, "/content/visclick"], check=True)
else:
    subprocess.run("cd /content/visclick && git pull --rebase", shell=True, check=False)
%cd /content/visclick


In [ ]:
import subprocess
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
        print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception as e:
    print("GPU check:", e)


In [ ]:
import os
DRIVE = "/content/drive/MyDrive/visclick"
os.makedirs(DRIVE, exist_ok=True)
for sub in (
    "data/raw", "data/clay", "data/vins", "data/webui", "data/ui_vision",
    "data/unified", "data/source_train", "data/desktop_labeled", "data/desktop_test",
    "weights/baseline_source", "weights/experiments", "videos",
):
    os.makedirs(os.path.join(DRIVE, sub), exist_ok=True)
print("DRIVE =", DRIVE)


In [ ]:
!pip install -q ultralytics==8.* opencv-python "huggingface_hub[cli]>=0.20" tqdm matplotlib pandas


## 01 — EDA (exploratory data analysis)

Expects at least one of: `DRIVE/data/unified`, `DRIVE/data/source_train`, or YOLO `images/`+`labels/` with nested train/val.

Saves figures under the repo: `reports/figures/eda/`.


In [ ]:
import os, glob, random, collections
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2

DRIVE = "/content/drive/MyDrive/visclick"
REPO = "/content/visclick"
OUT = os.path.join(REPO, "reports", "figures", "eda")
os.makedirs(OUT, exist_ok=True)

# Prefer user-assembled source_train; else try unified
candidates = [
    os.path.join(DRIVE, "data", "source_train"),
    os.path.join(DRIVE, "data", "unified"),
]
root = next((p for p in candidates if os.path.isdir(os.path.join(p, "labels", "train")) or os.path.isdir(os.path.join(p, "train", "labels"))), None)
if root is None:
    # YOLO layout: dataset/images/train and dataset/labels/train
    for p in candidates:
        if p and os.path.isdir(p):
            for images_dir in [os.path.join(p, "images", "train"), os.path.join(p, "train", "images")]:
                if os.path.isdir(images_dir):
                    root = p
                    break
        if root:
            break

if not root or not os.path.isdir(root or ""):
    print("No dataset found under", candidates, "- run 00 and assemble source_train, or download Zenodo unified first.")
else:
    print("Using root:", root)
    label_files = []
    for pat in [os.path.join(root, "labels", "train", "*.txt"), os.path.join(root, "train", "labels", "*.txt")]:
        label_files.extend(glob.glob(pat))
    if not label_files:
        label_files = glob.glob(os.path.join(root, "**", "labels", "*.txt"), recursive=True)[:5000]
    class_counts = collections.Counter()
    widths, heights = [], []
    for lf in label_files[:5000]:
        try:
            h, w = 0, 0
            im_base = os.path.splitext(os.path.basename(lf))[0]
            for sub in ["images/train", "train/images", "images"]:
                for ext in (".png", ".jpg", ".jpeg"):
                    ip = os.path.join(root, sub, im_base + ext)
                    if os.path.isfile(ip):
                        im = cv2.imread(ip)
                        if im is not None:
                            h, w = im.shape[:2]
                            break
        except Exception:
            pass
        with open(lf) as f:
            for line in f:
                parts = line.split()
                if len(parts) >= 5:
                    class_counts[int(parts[0])] += 1
        if w and h:
            widths.append(w)
            heights.append(h)
    if class_counts:
        plt.figure(figsize=(8, 4))
        cls_ids = sorted(class_counts.keys())
        plt.bar([str(x) for x in cls_ids], [class_counts[x] for x in cls_ids])
        plt.title("Instance count per class id (YOLO label files)")
        plt.xlabel("class id")
        p = os.path.join(OUT, "eda_class_balance.png")
        plt.savefig(p, dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved", p)
    if widths:
        plt.figure(figsize=(8, 4))
        plt.hist(np.array(widths), bins=20, alpha=0.5, label="W")
        plt.hist(np.array(heights), bins=20, alpha=0.5, label="H")
        plt.legend()
        plt.title("Image W/H in sampled set")
        p2 = os.path.join(OUT, "eda_resolutions.png")
        plt.savefig(p2, dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved", p2)
